# RNN model check

In [50]:
import sys
import numpy as np
import torch
sys.path.append('/home/mkhokhlo/projects/kotelnikov/scripts/scripts/vanilla')
sys.path.append('/home/mkhokhlo/projects/kotelnikov/scripts/scripts/')
from torch.utils.data import  Dataset, DataLoader
import torch.nn as nn

In [51]:
data_path = '/home/mkhokhlo/projects/kotelnikov/scripts/scripts/data/data_lstm_august24_PDET_left_right.csv'
int2feat ={0: 'ampl', 1: 'BandwidthesHz', 2: 'Bottom', 3: 'flash_periods', 4: 'flash_seconds', 5: 'frac_band', 6: 'freq', 7: 'phase', 8: 'Psi0',
           9: 'top', 10: 'time_sincelast'}

In [60]:
int2feat_all ={0: 'ampl', 1: 'BandwidthesHz', 2: 'Bottom', 3: 'flash_periods', 4: 'flash_seconds', 5: 'frac_band', 6: 'freq', 7:'left', 
                8: 'phase', 9: 'Psi0', 10: 'right',11: 'timeSigal', 12 :'top', 13: 'time_sincelast'}
# ampl
# BandwidthesHz
# Bottom
# flash_periods
# flash_seconds
# frac_band
# freq
# left
# phase
# Psi0
# right
# timeSignal
# top
# time_sincelast


In [52]:
from rnn import SimpleRNN, GaitDataset
from feature_selection import get_feature

In [61]:
current_feat = range(14)#[0,1,2,3,4,5,6,8,9,12,13]
num_feat = len(current_feat)
# Initialize lists to store the features and labels
PDL_features = [0]*num_feat
ETL_features = [0]*num_feat    

#     All but left, right, timeSignal.    
for i, j in enumerate(current_feat):
    Xf1_PDL, Xf1_ETL = get_feature(j,data_path= data_path,  min_len_established = 600, skip_patients = [55, 70,71,72,73,74,75,76])
    Xf1_PDL = np.array(Xf1_PDL)  # Shape: (14, 600)
    Xf1_ETL = np.array(Xf1_ETL)  # Shape: (14, 600)
    #print(Xf1_ETL.shape)
    PDL_features[i]= Xf1_PDL
    ETL_features[i]= Xf1_ETL         

PDL_features = np.array(PDL_features)  # Shape (14, 14, 600)
PDL_features = np.transpose(PDL_features, (1, 0, 2))  # Shape (14, 14, 600)


# Reshape ETL_features to (21, 14, 600)
ETL_features = np.array(ETL_features)  # Shape (14, 21, 600)
ETL_features = np.transpose(ETL_features, (1, 0, 2))  # Shape (21, 14, 600)

assert(PDL_features.shape == ETL_features.shape)


In [62]:
#min max normalization for the features for TRAIN
train_indices = range(14)
all_train = np.concatenate((PDL_features, ETL_features), axis=0) 

all_train =torch.tensor(all_train) #N train, 14,600
# Step 1: Reshape to combine patients and time dimensions for feature normalization
reshaped_features = all_train.permute(1, 0, 2).reshape(num_feat, -1)  # Shape: (14, N train *600)

# Step 2: Compute min and max values for each feature
min_values = reshaped_features.min(dim=1, keepdim=True).values  # Shape: (14, 1)
max_values = reshaped_features.max(dim=1, keepdim=True).values  # Shape: (14, 1)

normalized_features_train  = (reshaped_features - min_values) / (max_values - min_values + 1e-8)  # Avoid division by zero
normalized_features_train = normalized_features_train.reshape(num_feat, len(train_indices)*2, 600) 
normalized_features_train  = normalized_features_train.permute(1, 0, 2) # reshape back



PDL_train = normalized_features_train[:len(train_indices),:,:]
ETL_train =  normalized_features_train[len(train_indices):,:,:]

# Create dataset and dataloader
gait_dataset_train = GaitDataset(PDL_train, ETL_train)
dataloader = DataLoader(gait_dataset_train, batch_size=4, shuffle=True)

In [72]:
# Model parameters
input_size = num_feat  # Number of features
hidden_size = 8  # Hidden size for RNN
output_size = 1   # Binary classification (PDL or ETL)
num_epochs = 200

model = SimpleRNN(input_size, hidden_size, output_size)


    # Training setup
criterion = nn.BCELoss()  # Binary Cross Entropy Loss for binary classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# Training loop

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
model.to(device)
model.train()
for epoch in range(num_epochs):
            total_loss = 0.0
            correct_predictions = 0
            total_samples = 0
            
            for inputs, labels in dataloader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                optimizer.zero_grad()
                
                # Forward pass
                outputs = model(inputs)
                loss = criterion(outputs.squeeze(), labels.float())
                
                # Backward and optimize
                loss.backward()
                optimizer.step()
                
                # Track the total loss
                total_loss += loss.item() * inputs.size(0)
                
                # Convert probabilities to binary predictions (0 or 1)
                predicted = (outputs.squeeze() >= 0.5).long()
                
                # Track the number of correct predictions
                correct_predictions += (predicted == labels).sum().item()
                total_samples += labels.size(0)
            
            # Calculate average loss and accuracy
            avg_loss = total_loss / total_samples
            accuracy = correct_predictions / total_samples * 100
            
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")


Epoch [1/200], Loss: 0.7193, Accuracy: 42.86%
Epoch [2/200], Loss: 0.7189, Accuracy: 42.86%
Epoch [3/200], Loss: 0.7185, Accuracy: 42.86%
Epoch [4/200], Loss: 0.7181, Accuracy: 42.86%
Epoch [5/200], Loss: 0.7177, Accuracy: 42.86%
Epoch [6/200], Loss: 0.7173, Accuracy: 39.29%
Epoch [7/200], Loss: 0.7169, Accuracy: 39.29%
Epoch [8/200], Loss: 0.7165, Accuracy: 39.29%
Epoch [9/200], Loss: 0.7162, Accuracy: 39.29%
Epoch [10/200], Loss: 0.7158, Accuracy: 39.29%
Epoch [11/200], Loss: 0.7154, Accuracy: 39.29%
Epoch [12/200], Loss: 0.7150, Accuracy: 39.29%
Epoch [13/200], Loss: 0.7147, Accuracy: 39.29%
Epoch [14/200], Loss: 0.7143, Accuracy: 39.29%
Epoch [15/200], Loss: 0.7140, Accuracy: 39.29%
Epoch [16/200], Loss: 0.7136, Accuracy: 39.29%
Epoch [17/200], Loss: 0.7133, Accuracy: 39.29%
Epoch [18/200], Loss: 0.7130, Accuracy: 39.29%
Epoch [19/200], Loss: 0.7126, Accuracy: 39.29%
Epoch [20/200], Loss: 0.7123, Accuracy: 39.29%
Epoch [21/200], Loss: 0.7120, Accuracy: 39.29%
Epoch [22/200], Loss: 

In [73]:
dataloader = DataLoader(gait_dataset_train, batch_size=28, shuffle=False)

for X_val, y_val in dataloader: 
    print(X_val.shape)
X_val.requires_grad = True  # Enable gradient tracking for the inputs
outputs = model(X_val)
loss = criterion(outputs.squeeze(), y_val.float())
loss.backward()

# Compute average absolute gradient for each feature
feature_gradients = X_val.grad.abs().mean(dim=(0, 1))  # Average over batch and time dimensions
important_features = torch.argsort(feature_gradients, descending=True).tolist()
print("Feature Importance (from most to least):")
for num in important_features:
    if num in int2feat_all:
        feature_name = int2feat_all[num]
        gradient_value = feature_gradients[num].item()
        print(f"{feature_name}: Gradient = {gradient_value:.6f}")

torch.Size([28, 600, 14])
Feature Importance (from most to least):
time_sincelast: Gradient = 0.000027
phase: Gradient = 0.000025
flash_periods: Gradient = 0.000021
left: Gradient = 0.000020
Bottom: Gradient = 0.000015
flash_seconds: Gradient = 0.000012
frac_band: Gradient = 0.000012
top: Gradient = 0.000012
Psi0: Gradient = 0.000009
right: Gradient = 0.000009
BandwidthesHz: Gradient = 0.000008
timeSigal: Gradient = 0.000007
ampl: Gradient = 0.000006
freq: Gradient = 0.000005


In [71]:
## Code to Verify Feature Normalization:

# Assuming X_val is the tensor of your features
# X_val has the shape (batch_size, sequence_length, num_features)

# Calculate mean and standard deviation for each feature
feature_means = X_val.mean(dim=(0, 1))  # Mean across batch and time dimensions
feature_stds = X_val.std(dim=(0, 1))    # Standard deviation across batch and time dimensions

# Print results for inspection
for i, (mean, std) in enumerate(zip(feature_means.tolist(), feature_stds.tolist())):
    print(f"Feature {i} - Mean: {mean:.5f}, Std: {std:.5f}")

# Check if features are approximately normalized (mean ~ 0, std ~ 1)
is_normalized = all(abs(mean) < 1e-5 and abs(std - 1) < 1e-2 for mean, std in zip(feature_means, feature_stds))
if is_normalized:
    print("Features are correctly normalized!")
else:
    print("Features are not properly normalized. Consider applying normalization.")

Feature 0 - Mean: 0.00710, Std: 0.03873
Feature 1 - Mean: 0.17115, Std: 0.11410
Feature 2 - Mean: 0.45620, Std: 0.26271
Feature 3 - Mean: 0.07186, Std: 0.04765
Feature 4 - Mean: 0.02657, Std: 0.05333
Feature 5 - Mean: 0.18118, Std: 0.06836
Feature 6 - Mean: 0.47709, Std: 0.27572
Feature 7 - Mean: 0.33285, Std: 0.20549
Feature 8 - Mean: 0.49914, Std: 0.27072
Feature 9 - Mean: 0.50178, Std: 0.28947
Feature 10 - Mean: 0.33377, Std: 0.20556
Feature 11 - Mean: 0.33330, Std: 0.20552
Feature 12 - Mean: 0.48615, Std: 0.28136
Feature 13 - Mean: 0.10275, Std: 0.08264
Features are not properly normalized. Consider applying normalization.


In [49]:
# check for code for normalization
# Reshape to combine patients and time for normalization
print(all_train.shape)
reshaped_features = all_train.permute(1, 0, 2).reshape(num_feat, -1)  # after permutation, all_train.permute(1, 0, 2) changes the shape to (11, N_train, 600). Now, the features are in the first dimension, followed by the number of patients and the time dimension.

# Min-max normalization across features
min_values = reshaped_features.min(dim=1, keepdim=True).values  # Shape: (num_feat, 1)
max_values = reshaped_features.max(dim=1, keepdim=True).values  # Shape: (num_feat, 1)


# # Normalize
normalized_features_train = (reshaped_features - min_values) / (max_values - min_values + 1e-8)  # Avoid division by zero
print(normalized_features_train.shape)

# # Reshape back to the original structure (patients, features, time)
normalized_features_train = reshaped_features.reshape(num_feat, len(train_indices)*2,  600)
normalized_features_train  = normalized_features_train .permute(1, 0, 2)

print(normalized_features_train.shape)
# Check if two tensors are nearly equal
are_equal = torch.allclose(all_train, normalized_features_train, atol=1e-6)
# Print result
if are_equal:
    print("The two tensors are nearly equal.")
else:
    print("The two tensors are not equal.")

# # Split the normalized features into PDL and ETL datasets
# PDL_train = normalized_features_train[:len(train_indices), :, :]
# ETL_train = normalized_features_train[len(train_indices):, :, :]

# # Check the normalized shape
# print("Normalized train shape:", normalized_features_train.shape)

torch.Size([28, 11, 600])
torch.Size([11, 16800])
torch.Size([28, 11, 600])
The two tensors are nearly equal.
